# **Learning Roadmap Chatbot (offline)**

<br>

Learning a new skill online is overwhelming. A beginner searching for a topic like machine learning or web development is flooded with thousands of courses, videos, and blog posts, with no clear idea of what to study first or how topics connect. Many start with advanced material without prerequisites, jump between resources, lose motivation, and give up entirely. Existing roadmap guides are static, scattered across websites, and often require internet access or paid AI tools. Students, especially those with limited connectivity or budgets, urgently need a simple, free, structured way to know exactly what to learn and in what order.

<br>

### **Solution:**

We built an offline chatbot using Gradio that generates a complete learning roadmap. The user types a topic, such as “Machine Learning” or “I want to learn web development”, and the bot replies with ordered steps: foundations, core skills, hands-on practice, and advanced concepts, each listing what to study. A built-in knowledge base covers popular topics, while fuzzy matching handles typos. For unfamiliar topics, a general four-stage template still guides the learner. Because it uses no external models or APIs, it is free, fast, private, and works without internet once launched. New topics can be added by editing one dictionary.

In [6]:
!pip install -q "gradio>=5,<6"

In [7]:
import re
import difflib
import gradio as gr

In [8]:
# Each topic: keywords that trigger it -> ordered list of stages
ROADMAPS = {
    "Web Development": {
        "keywords": ["web development", "web dev", "frontend", "full stack", "fullstack", "website"],
        "stages": [
            ("Foundations", ["HTML", "CSS (Flexbox, Grid)", "Git & GitHub", "Basic JavaScript"]),
            ("Core Skills", ["Advanced JavaScript (ES6+, async/await)", "A framework (React)", "REST APIs & fetch"]),
            ("Backend", ["Node.js + Express", "A database (PostgreSQL or MongoDB)", "Authentication (JWT)"]),
            ("Advanced", ["TypeScript", "Testing", "Deployment (Vercel, Docker)", "System design basics"]),
        ],
    },
    "Machine Learning": {
        "keywords": ["machine learning", "ml", "ai", "artificial intelligence", "deep learning"],
        "stages": [
            ("Foundations", ["Python", "NumPy, Pandas, Matplotlib", "Linear algebra, probability, statistics"]),
            ("Classical ML", ["Regression & classification", "Decision trees, SVM, k-NN", "scikit-learn", "Model evaluation"]),
            ("Deep Learning", ["Neural network basics", "PyTorch or TensorFlow", "CNNs and RNNs", "Transformers"]),
            ("Advanced", ["MLOps & deployment", "Research papers", "Kaggle competitions"]),
        ],
    },
    "Data Structures & Algorithms": {
        "keywords": ["data structures", "algorithms", "dsa", "competitive programming", "leetcode"],
        "stages": [
            ("Foundations", ["One language (C++, Java, or Python)", "Time & space complexity (Big-O)", "Arrays & strings"]),
            ("Core Structures", ["Linked lists", "Stacks & queues", "Hash maps", "Trees & BSTs"]),
            ("Core Algorithms", ["Sorting & searching", "Recursion & backtracking", "Graphs (BFS, DFS)", "Heaps"]),
            ("Advanced", ["Dynamic programming", "Greedy algorithms", "Tries, segment trees", "Mock contests"]),
        ],
    },
    "Data Science": {
        "keywords": ["data science", "data analysis", "data analyst", "analytics"],
        "stages": [
            ("Foundations", ["Python", "SQL", "Statistics basics", "Excel"]),
            ("Core Skills", ["Pandas & NumPy", "Data cleaning", "Visualization (Matplotlib, Seaborn)", "EDA"]),
            ("Modeling", ["Hypothesis testing", "scikit-learn", "Feature engineering"]),
            ("Advanced", ["Dashboards (Power BI / Tableau)", "Big data basics", "End-to-end portfolio projects"]),
        ],
    },
    "Python": {
        "keywords": ["python"],
        "stages": [
            ("Basics", ["Variables, data types, operators", "Conditionals & loops", "Functions"]),
            ("Intermediate", ["Lists, dicts, sets, tuples", "File handling", "Modules & packages", "OOP"]),
            ("Practical", ["Virtual environments & pip", "APIs & requests", "Error handling & testing"]),
            ("Advanced", ["Decorators & generators", "Async programming", "A framework (Flask / FastAPI)"]),
        ],
    },
    "Cybersecurity": {
        "keywords": ["cybersecurity", "cyber security", "ethical hacking", "security", "hacking"],
        "stages": [
            ("Foundations", ["Networking (TCP/IP, DNS, HTTP)", "Linux command line", "Basic scripting (Python/Bash)"]),
            ("Core Skills", ["Cryptography basics", "Web vulnerabilities (OWASP Top 10)", "Security tools (Wireshark, Nmap)"]),
            ("Hands-on", ["CTF challenges", "TryHackMe / Hack The Box", "Vulnerability scanning"]),
            ("Advanced", ["Penetration testing", "Incident response", "Certifications (Security+, CEH)"]),
        ],
    },
}

In [9]:
def find_topic(message):
    """Match the message to a known topic. Returns topic name or None."""
    text = message.lower()
    # 1) exact keyword match (whole words)
    for name, data in ROADMAPS.items():
        for kw in data["keywords"]:
            if re.search(rf"\b{re.escape(kw)}\b", text):
                return name
    # 2) fuzzy match to handle typos
    all_kws = {kw: name for name, d in ROADMAPS.items() for kw in d["keywords"]}
    for word in re.findall(r"[a-z]+(?: [a-z]+)?", text):
        close = difflib.get_close_matches(word, all_kws, n=1, cutoff=0.8)
        if close:
            return all_kws[close[0]]
    return None


def clean_topic(message):
    """Pull the topic out of phrases like 'I want to learn X'."""
    text = re.sub(r"(?i)\b(i want to|how do i|how to|please|roadmap|learn|study|for|on|about|a|the)\b", " ", message)
    return " ".join(text.split()).strip(" ?.!") or "this topic"


def format_roadmap(title, stages):
    out = f"## Roadmap: {title}\n"
    for i, (stage, items) in enumerate(stages, 1):
        out += f"\n### Step {i}: {stage}\n"
        out += "\n".join(f"{j}. {item}" for j, item in enumerate(items, 1)) + "\n"
    out += "\n**Tip:** build a small project at the end of every step."
    return out


def generic_roadmap(topic):
    stages = [
        ("Understand the Basics", [f"What is {topic} and why it matters", "Key terms and core concepts", "A beginner course or tutorial"]),
        ("Learn the Fundamentals", [f"Main building blocks of {topic}", "Read the official docs or a standard textbook", "Practice with small exercises"]),
        ("Build Projects", ["Start with a simple project", "Increase difficulty gradually", "Share your work and get feedback"]),
        ("Go Deeper", ["Advanced topics and best practices", "Join communities / forums", "Teach or write about what you learned"]),
    ]
    return format_roadmap(topic.title(), stages)


def roadmap_bot(message, history):
    topic = find_topic(message)
    if topic:
        return format_roadmap(topic, ROADMAPS[topic]["stages"])
    return ("I don't have a detailed roadmap for that yet, so here is a general one.\n\n"
            + generic_roadmap(clean_topic(message)))

In [10]:
demo = gr.ChatInterface(
    fn=roadmap_bot,
    type="messages",
    title="Learning Roadmap Bot",
    description="Tell me what you want to learn and I'll give you a step-by-step roadmap.",
    examples=["Machine Learning", "Web Development", "Data Structures & Algorithms", "I want to learn cybersecurity"],
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c60656c88cc363c3fa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
